##Multi-Domain AI Model Recommendation Decision Support System

In [ ]:
import subprocess, sys, re

def _ensure(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

for pkg, mod in [("sentence-transformers", "sentence_transformers"),
                  ("gradio", "gradio"),
                  ("matplotlib", "matplotlib"),
                  ("markdown", "markdown")]:
    _ensure(pkg, mod)

import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

semantic_model = SentenceTransformer('all-MiniLM-L6-v2')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. FUZZY LOGIC CORE  (domain-agnostic)
# ─────────────────────────────────────────────────────────────────────────────

def triangular_mf(x: float, a: float, b: float, c: float) -> float:
    if a == b:
        return 1.0 if x <= b else max(0.0, (c - x) / (c - b))
    if b == c:
        return 0.0 if x <= a else (1.0 if x >= b else (x - a) / (b - a))
    if x <= a or x >= c:
        return 0.0
    if a < x <= b:
        return (x - a) / (b - a)
    return (c - x) / (c - b)


def fuzzify_weights(raw_scores: dict, normalize: bool = True) -> dict:
    """Center-of-Gravity defuzzification + normalization to sum to 1."""
    fuzzy_weights = {}
    for criterion, score in raw_scores.items():
        if score == 0.0:
            fuzzy_weights[criterion] = 0.0
            continue
        low    = triangular_mf(score, 0.0, 0.0, 0.5)
        medium = triangular_mf(score, 0.2, 0.5, 0.8)
        high   = triangular_mf(score, 0.5, 1.0, 1.0)
        total = low + medium + high
        centroid = (0.0 * low + 0.5 * medium + 1.0 * high) / total if total > 0 else 0.0
        fuzzy_weights[criterion] = round(centroid, 4)
    if normalize:
        s = sum(fuzzy_weights.values())
        if s > 0:
            fuzzy_weights = {c: round(w / s, 4) for c, w in fuzzy_weights.items()}
    return fuzzy_weights


def fuzzy_soft_decision(dataset: pd.DataFrame, fuzzy_weights: dict) -> pd.DataFrame:
    scores = {
        model: round(
            sum(dataset.loc[model, c] * fuzzy_weights.get(c, 0.0) for c in dataset.columns), 4
        )
        for model in dataset.index
    }
    ranking = (
        pd.DataFrame(scores.items(), columns=["Model", "Score"])
        .sort_values("Score", ascending=False)
        .reset_index(drop=True)
    )
    ranking.index += 1
    return ranking


def score_label(s: float) -> str:
    if s >= 0.90: return "exceptional"
    if s >= 0.80: return "strong"
    if s >= 0.70: return "good"
    if s >= 0.60: return "moderate"
    return "below-average"

# DATA PROVENANCE NOTE (applies to every dataset below):
   - For the Text/Code domain, criteria like Reasoning/Coding/Mathematics/
     Multilingual are REAL published benchmark percentages (GPQA Diamond,
     SWE-bench Verified, MATH-500/AIME, MGSM) simply divided by 100 to land
     in the [0,1] range used throughout this system - that's a direct,
     not-estimated conversion.
   - Criteria with no single universal public benchmark (Summarization,
     Long_Context tier, Creativity, Cost_Efficiency, Speed, Safety for
     text; almost ALL criteria for Image/Video/Speech domains, since those
     fields don't yet have GPQA-style standardized leaderboards) are
     QUALITATIVE ESTIMATES, normalized onto a 0-1 scale by hand based on
     comparison/review sources cited in each domain's BENCHMARK_SOURCE
     dict below. These should be treated as directional signals, not
     precise measurements - re-verify before using this for a real
     purchasing/deployment decision.
   Last checked: late June 2026.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. DOMAIN DEFINITIONS
# ─────────────────────────────────────────────────────────────────────────────
def _make_domain(name, routing_description, criteria, criterion_descriptions,
                  benchmark_source, raw_scores):
    dataset = pd.DataFrame.from_dict(raw_scores, orient="index", columns=criteria)
    criterion_embeddings = {c: semantic_model.encode(d) for c, d in criterion_descriptions.items()}
    domain_embedding = semantic_model.encode(routing_description)
    return {
        "name": name,
        "routing_description": routing_description,
        "domain_embedding": domain_embedding,
        "criteria": criteria,
        "criterion_descriptions": criterion_descriptions,
        "criterion_embeddings": criterion_embeddings,
        "benchmark_source": benchmark_source,
        "dataset": dataset,
    }


# ── DOMAIN: Text / Code LLMs ────────────────────────────────────────────────
TEXT_CRITERIA = [
    "Reasoning", "Coding", "Mathematics", "Summarization",
    "Multilingual", "Long_Context", "Creativity",
    "Cost_Efficiency", "Speed", "Safety",
]
TEXT_CRITERION_DESCRIPTIONS = {
    "Reasoning": "logical reasoning analytical thinking problem solving decision making chain of thought step by step inference deduction",
    "Coding": "programming software engineering debugging code generation refactoring code review automated software development SWE bug fix",
    "Mathematics": "mathematics numerical computation algebra calculus arithmetic equations quantitative reasoning olympiad statistics",
    "Summarization": "summarising condensing long documents text understanding information extraction document question answering reading comprehension",
    "Multilingual": "translation multiple languages multilingual cross lingual Arabic French Chinese Urdu Hindi Spanish language support",
    "Long_Context": "processing very long documents legal contracts research papers large context windows book length input needle in haystack retrieval",
    "Creativity": "creative writing storytelling poetry fiction narrative brainstorming content generation imagination originality",
    "Cost_Efficiency": "low cost affordable budget pricing cheap economical inexpensive tokens per dollar value for money",
    "Speed": "fast response low latency quick generation rapid inference real time streaming tokens per second throughput",
    "Safety": "safe trustworthy reliable low hallucination responsible AI aligned factual accurate honest non harmful ethical",
}
TEXT_BENCHMARK_SOURCE = {
    "Reasoning": "GPQA Diamond (direct %)", "Coding": "SWE-bench Verified (direct %)",
    "Mathematics": "MATH-500 / AIME (direct %)",
    "Summarization": "Expert evaluation / HELMET long-doc (qualitative estimate)",
    "Multilingual": "MGSM (direct %)",
    "Long_Context": "NIAH / context-window tier (qualitative estimate)",
    "Creativity": "LMSYS / Arena creative writing Elo (qualitative estimate)",
    "Cost_Efficiency": "Inverse pricing $/1M tokens (normalized estimate)",
    "Speed": "Tokens/second (normalized estimate)",
    "Safety": "Model card safety evaluation (qualitative estimate)",
}
TEXT_RAW_SCORES = {
    "GPT-4o":            [0.54, 0.33, 0.77, 0.82, 0.89, 0.85, 0.88, 0.42, 0.72, 0.88],
    "Claude Sonnet 4":   [0.78, 0.73, 0.90, 0.92, 0.91, 0.97, 0.91, 0.38, 0.65, 0.94],
    "Gemini 2.5 Pro":    [0.84, 0.63, 0.92, 0.90, 0.96, 0.99, 0.88, 0.45, 0.70, 0.89],
    "Gemini 2.5 Flash":  [0.74, 0.54, 0.88, 0.85, 0.93, 0.97, 0.83, 0.88, 0.95, 0.87],
    "DeepSeek V3":       [0.59, 0.42, 0.90, 0.83, 0.87, 0.84, 0.80, 0.97, 0.82, 0.80],
    "Llama 4":           [0.70, 0.33, 0.87, 0.81, 0.87, 0.90, 0.82, 0.95, 0.88, 0.83],
    "Mistral Large":     [0.56, 0.32, 0.84, 0.80, 0.84, 0.80, 0.80, 0.78, 0.82, 0.82],
    "Qwen 3":            [0.68, 0.49, 0.93, 0.85, 0.94, 0.93, 0.84, 0.90, 0.85, 0.84],
    "Claude Opus 4.8":   [0.94, 0.89, 0.95, 0.93, 0.95, 0.99, 0.85, 0.35, 0.55, 0.95],
    "GPT-5.5":           [0.93, 0.89, 0.94, 0.88, 0.90, 0.95, 0.82, 0.45, 0.65, 0.90],
    "Gemini 3.1 Pro":    [0.94, 0.81, 0.93, 0.85, 0.93, 0.99, 0.78, 0.55, 0.80, 0.85],
    "Grok 4":            [0.88, 0.75, 0.92, 0.75, 0.80, 0.85, 0.80, 0.50, 0.75, 0.65],
}

In [ ]:
# ── DOMAIN: Image Generation ────────────────────────────────────────────────
IMAGE_CRITERIA = [
    "Prompt_Adherence", "Photorealism", "Artistic_Quality", "Text_Rendering",
    "Editing_Capability", "Cost_Efficiency", "Speed", "Commercial_Safety",
]
IMAGE_CRITERION_DESCRIPTIONS = {
    "Prompt_Adherence": "following complex detailed instructions accurately precise prompt understanding multi clause descriptions",
    "Photorealism": "photorealistic realistic photography lifelike human faces skin texture lighting natural images",
    "Artistic_Quality": "artistic aesthetic stylized creative painterly illustration distinctive visual style concept art",
    "Text_Rendering": "rendering legible text typography logos posters words letters signage in the image",
    "Editing_Capability": "image editing inpainting outpainting generative fill modifying existing images touch ups",
    "Cost_Efficiency": "cheap affordable low cost per image budget friendly free open source self hosted",
    "Speed": "fast quick generation time rapid turnaround low latency image creation",
    "Commercial_Safety": "commercially safe licensed training data copyright safe IP indemnification enterprise brand use",
}
# All scores below are QUALITATIVE ESTIMATES normalized to 0-1 by hand from
# June 2026 comparison roundups (llm-stats.com Image Arena, gradually.ai,
# getaiperks.com, diyai.io, cliprise.app, pixazo.ai, morphed.app). Image
# generation has no single universal numeric benchmark equivalent to
# GPQA/SWE-bench - these are directional, not precise measurements.
IMAGE_BENCHMARK_SOURCE = {
    "Prompt_Adherence": "Image Arena blind human votes / GenEval-style testing (estimate)",
    "Photorealism": "Independent comparison testing (estimate)",
    "Artistic_Quality": "Image Arena Elo / style-range scoring (estimate)",
    "Text_Rendering": "In-image typography accuracy testing (estimate)",
    "Editing_Capability": "Inpainting/outpainting/editing comparison reviews (estimate)",
    "Cost_Efficiency": "Per-image API pricing, inverse (estimate)",
    "Speed": "Generation time per image, inverse (estimate)",
    "Commercial_Safety": "Training-data licensing / IP indemnification policy (estimate)",
}
IMAGE_RAW_SCORES = {
    "GPT Image 2":          [0.96, 0.85, 0.75, 0.90, 0.85, 0.55, 0.65, 0.70],
    "Midjourney V7":        [0.70, 0.80, 0.98, 0.55, 0.60, 0.60, 0.70, 0.65],
    "FLUX.2 Pro":           [0.82, 0.96, 0.80, 0.65, 0.75, 0.85, 0.90, 0.65],
    "Imagen 4":             [0.85, 0.93, 0.70, 0.95, 0.70, 0.60, 0.80, 0.85],
    "Adobe Firefly":        [0.78, 0.75, 0.70, 0.70, 0.90, 0.55, 0.75, 0.98],
    "Stable Diffusion 3.5": [0.65, 0.75, 0.75, 0.55, 0.70, 0.97, 0.70, 0.50],
    "Ideogram 3":           [0.75, 0.70, 0.75, 0.97, 0.65, 0.70, 0.80, 0.70],
}

In [ ]:
# ── DOMAIN: Video Generation ────────────────────────────────────────────────
VIDEO_CRITERIA = [
    "Prompt_Adherence", "Motion_Quality", "Visual_Fidelity",
    "Consistency", "Cost_Efficiency", "Speed", "Max_Duration",
]
VIDEO_CRITERION_DESCRIPTIONS = {
    "Prompt_Adherence": "following video prompt instructions camera movement scene description accuracy",
    "Motion_Quality": "realistic natural motion physics movement fluid animation no jitter",
    "Visual_Fidelity": "high resolution sharp detailed cinematic visual quality video frames",
    "Consistency": "consistent characters objects across frames no flickering temporal stability",
    "Cost_Efficiency": "cheap affordable low cost per second per clip budget friendly video generation",
    "Speed": "fast quick video generation time rapid turnaround",
    "Max_Duration": "longer video clips extended duration length of generated video",
}
# QUALITATIVE ESTIMATES based on general industry positioning as of mid-2026
# (Sora 2, Veo 3, Runway Gen-4, Kling 2.5, Pika 2). Video benchmarking
# (VBench etc.) is the least standardized of the four domains here - treat
# as a starting point, not a precise leaderboard.
VIDEO_BENCHMARK_SOURCE = {
    "Prompt_Adherence": "VBench-style prompt-following evaluation (estimate)",
    "Motion_Quality": "VBench motion smoothness / physics evaluation (estimate)",
    "Visual_Fidelity": "Independent resolution/quality comparison testing (estimate)",
    "Consistency": "VBench temporal consistency evaluation (estimate)",
    "Cost_Efficiency": "Per-second/per-clip API pricing, inverse (estimate)",
    "Speed": "Generation time per clip, inverse (estimate)",
    "Max_Duration": "Published max clip length per provider (direct spec)",
}
VIDEO_RAW_SCORES = {
    "Sora 2":       [0.88, 0.92, 0.90, 0.85, 0.40, 0.50, 0.70],
    "Veo 3":        [0.90, 0.88, 0.88, 0.85, 0.55, 0.60, 0.65],
    "Runway Gen-4": [0.80, 0.82, 0.82, 0.80, 0.55, 0.65, 0.60],
    "Kling 2.5":    [0.78, 0.85, 0.80, 0.78, 0.75, 0.75, 0.65],
    "Pika 2":       [0.65, 0.70, 0.65, 0.65, 0.90, 0.90, 0.40],
}

In [ ]:
# ── DOMAIN: Speech / Voice (TTS & voice cloning) ────────────────────────────
SPEECH_CRITERIA = [
    "Naturalness", "Voice_Cloning", "Multilingual", "Latency",
    "Cost_Efficiency", "Emotional_Range",
]
SPEECH_CRITERION_DESCRIPTIONS = {
    "Naturalness": "natural human sounding voice realistic speech no robotic artifacts",
    "Voice_Cloning": "cloning a specific voice custom voice creation from samples voice replication",
    "Multilingual": "speaking multiple languages accents wide language support text to speech",
    "Latency": "low latency real time conversation fast response voice assistant streaming speech",
    "Cost_Efficiency": "cheap affordable low cost per character per minute budget friendly speech generation",
    "Emotional_Range": "emotional expressive tone variation acting voice performance nuance",
}
# QUALITATIVE ESTIMATES from provider documentation and TTS comparison
# reviews as of mid-2026. "Multilingual" is closer to a direct spec
# (published supported-language count, normalized); the rest are estimated.
SPEECH_BENCHMARK_SOURCE = {
    "Naturalness": "MOS-style naturalness listening tests (estimate)",
    "Voice_Cloning": "Voice cloning fidelity comparison reviews (estimate)",
    "Multilingual": "Published supported-language count (direct, normalized)",
    "Latency": "Time-to-first-audio benchmark, inverse (estimate)",
    "Cost_Efficiency": "Per-character/per-minute pricing, inverse (estimate)",
    "Emotional_Range": "Expressiveness / acting-range comparison reviews (estimate)",
}
SPEECH_RAW_SCORES = {
    "ElevenLabs v3":       [0.95, 0.95, 0.85, 0.70, 0.45, 0.95],
    "OpenAI Realtime TTS": [0.85, 0.30, 0.80, 0.92, 0.65, 0.75],
    "Google Gemini Audio": [0.82, 0.50, 0.95, 0.80, 0.65, 0.65],
    "PlayHT 3.0":          [0.78, 0.85, 0.70, 0.75, 0.80, 0.70],
}

DOMAINS = {
    "Text / Code LLMs": _make_domain(
        "Text / Code LLMs",
        "writing code debugging programming software engineering answering questions "
        "reasoning summarizing documents translating languages writing essays stories "
        "chatbot conversations analysis math problems text generation database api "
        "backend frontend web development",
        TEXT_CRITERIA, TEXT_CRITERION_DESCRIPTIONS, TEXT_BENCHMARK_SOURCE, TEXT_RAW_SCORES,
    ),
    "Image Generation": _make_domain(
        "Image Generation",
        "generating creating images pictures artwork illustrations photos drawings "
        "from text prompts image generation text to image digital art logos posters "
        "photorealistic images graphic design",
        IMAGE_CRITERIA, IMAGE_CRITERION_DESCRIPTIONS, IMAGE_BENCHMARK_SOURCE, IMAGE_RAW_SCORES,
    ),
    "Video Generation": _make_domain(
        "Video Generation",
        "generating creating videos clips animations from text prompts text to video "
        "AI video generation motion graphics short films cinematic footage",
        VIDEO_CRITERIA, VIDEO_CRITERION_DESCRIPTIONS, VIDEO_BENCHMARK_SOURCE, VIDEO_RAW_SCORES,
    ),
    "Speech / Voice (TTS)": _make_domain(
        "Speech / Voice (TTS)",
        "generating speech voice audio text to speech voice cloning synthesizing "
        "spoken audio narration podcasts voiceover audiobook reading aloud",
        SPEECH_CRITERIA, SPEECH_CRITERION_DESCRIPTIONS, SPEECH_BENCHMARK_SOURCE, SPEECH_RAW_SCORES,
    ),
}


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. DOMAIN ROUTER (hybrid: keyword pass first, semantic-embedding fallback)
# ─────────────────────────────────────────────────────────────────────────────

DOMAIN_KEYWORDS = {
    "Text / Code LLMs": [
        "code", "coding", "programming", "software", "developer", "python", "java",
        "javascript", "c++", "c#", "php", "bug", "debug", "debugging", "algorithm",
        "database", "sql", "nosql", "api", "rest api", "backend", "frontend",
        "mobile app", "web app", "integration", "system design", "microservices",
        "docker", "kubernetes", "cloud", "aws", "azure", "machine learning",
        "deep learning", "ai model", "llm", "chatbot", "assistant", "reasoning",
        "mathematics", "math", "equation", "essay", "document", "research paper",
        "translation", "summarization", "software engineering",
    ],
    "Image Generation": [
        "image", "photo", "picture", "drawing", "illustration", "art", "painting",
        "portrait", "anime", "logo", "poster", "flyer", "graphic", "design",
        "photorealistic", "concept art", "text to image", "image generation",
        "stable diffusion", "midjourney", "flux", "imagen", "firefly",
    ],
    "Video Generation": [
        "video", "animation", "movie", "film", "clip", "cinematic", "motion",
        "camera", "fps", "scene", "transition", "frame", "render", "text to video",
        "video generation", "runway", "sora", "veo", "kling", "pika",
    ],
    "Speech / Voice (TTS)": [
        "voice", "speech", "audio", "text to speech", "tts", "voice clone",
        "voice cloning", "podcast", "narration", "speaker", "pronunciation",
        "microphone", "realtime voice", "elevenlabs", "playht",
    ],
}

KEYWORD_WEIGHTS = {
    "software engineering": 4, "programming": 4, "coding": 4, "developer": 4,
    "debugging": 4, "integration": 4, "system design": 4, "microservices": 4,
    "python": 3, "java": 3, "javascript": 3, "c++": 3, "c#": 3, "api": 3,
    "rest api": 3, "database": 3, "sql": 3, "docker": 3, "kubernetes": 3,
    "cloud": 3, "aws": 3, "azure": 3, "algorithm": 2, "math": 2, "reasoning": 2,
    "image": 4, "photo": 4, "picture": 4, "illustration": 4, "art": 4, "logo": 4,
    "poster": 4, "photorealistic": 4,
    "video": 4, "animation": 4, "movie": 4, "film": 4, "clip": 4, "cinematic": 4,
    "frame": 3, "motion": 3,
    "voice": 4, "speech": 4, "audio": 4, "tts": 4, "voice cloning": 4,
    "podcast": 3, "narration": 3,
}


def _keyword_in_text(word: str, text: str) -> bool:
    """
    FIX: re's \\b word-boundary fails on tokens ending in punctuation like
    "c++" or "c#", because \\b only fires at a transition between a word
    character and a non-word character - and the LAST character of "c++"
    is itself a non-word character, so there's never a boundary there.
    This uses a custom boundary based on alphanumeric characters only,
    which correctly handles tokens containing +, #, or other symbols.
    """
    pattern = r'(?<![A-Za-z0-9])' + re.escape(word) + r'(?![A-Za-z0-9])'
    return re.search(pattern, text) is not None


def route_domain(user_scenario: str):
    """Stage 1: keyword matching. Stage 2 (fallback): semantic similarity."""
    text = user_scenario.lower()

    keyword_scores = {}
    for domain, keywords in DOMAIN_KEYWORDS.items():
        score = 0
        for word in keywords:
            if _keyword_in_text(word, text):
                score += KEYWORD_WEIGHTS.get(word, 1)
        keyword_scores[domain] = score

    max_keyword = max(keyword_scores.values())

    if max_keyword >= 3:
        best_domain = max(keyword_scores, key=keyword_scores.get)
        total = sum(keyword_scores.values())
        similarities = {d: round(s / total if total else 0, 3) for d, s in keyword_scores.items()}
        return best_domain, similarities

    scenario_embedding = semantic_model.encode(user_scenario)
    semantic_scores = {
        name: round(float(cosine_similarity([scenario_embedding], [d["domain_embedding"]])[0][0]), 3)
        for name, d in DOMAINS.items()
    }
    best_domain = max(semantic_scores, key=semantic_scores.get)
    return best_domain, semantic_scores

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. CRITERION SELECTION (semantic similarity + keyword boost + RELATIVE cutoff)
# ─────────────────────────────────────────────────────────────────────────────

# Boost keywords per criterion. Reused across domains where criterion names
# match (Cost_Efficiency, Speed, Multilingual appear in several domains).
CRITERION_BOOST_KEYWORDS = {
    "Coding": ["code", "coding", "programming", "developer", "software", "debug", "debugging", "bug"],
    "Reasoning": ["reasoning", "logic", "logical", "problem solving", "analysis"],
    "Mathematics": ["math", "mathematics", "equation", "algebra", "calculus", "statistics"],
    "Summarization": ["summary", "summarize", "summarisation", "summarization"],
    "Multilingual": ["translate", "translation", "language", "arabic", "urdu", "french", "spanish", "multilingual"],
    "Long_Context": ["research paper", "book", "large document", "long context", "contract"],
    "Creativity": ["essay", "story", "creative", "poem", "brainstorm"],
    "Cost_Efficiency": ["cheap", "budget", "affordable", "low cost", "free", "cost effective", "cost-effective", "inexpensive", "economical"],
    "Speed": ["fast", "quick", "rapid", "low latency", "real time", "speed", "speedy"],
    "Safety": ["safe", "secure", "trustworthy"],
    "Prompt_Adherence": ["accurate", "precise", "follow instructions", "exact", "specific instructions"],
    "Photorealism": ["photorealistic", "realistic", "lifelike", "photo realistic"],
    "Artistic_Quality": ["artistic", "stylized", "aesthetic", "art style"],
    "Text_Rendering": ["text in image", "typography", "logo text", "readable text", "words in the image"],
    "Editing_Capability": ["edit", "inpaint", "outpaint", "modify image", "image editing"],
    "Commercial_Safety": ["commercial use", "licensed", "copyright safe", "brand safe"],
    "Motion_Quality": ["motion", "movement", "physics"],
    "Visual_Fidelity": ["resolution", "cinematic", "high quality video", "4k"],
    "Consistency": ["consistent characters", "no flickering", "consistency"],
    "Max_Duration": ["long video", "extended duration", "longer clip"],
    "Naturalness": ["natural voice", "human sounding", "natural sounding"],
    "Voice_Cloning": ["clone voice", "voice cloning", "custom voice"],
    "Latency": ["real time", "low latency", "instant response"],
    "Emotional_Range": ["expressive", "emotional", "emotion"],
}




    FIX: previously a FIXED ABSOLUTE threshold (e.g. ">= 0.60 after a +0.40
    keyword boost") was used to decide which criteria survive. That fails in
    two opposite ways: (1) without an exact keyword hit, almost nothing
    reaches 0.60, so genuinely relevant-but-unworded criteria get dropped -
    collapsing every scenario down to exactly 1 active criterion; or
    (2) with a low/no threshold, irrelevant criteria get force-included by
    top_k regardless of how weakly they actually matched.

    The fix: keep a small keyword boost (it correctly nudges the right
    criterion when you use an exact term like "fast" or "coding"), but
    decide which of the top_k candidates SURVIVE using a threshold that's
    RELATIVE TO THIS SCENARIO's own top score, not a fixed global number.
    A criterion survives if its score is at least `relative_floor` (55%)
    of the best-matching criterion's score in this specific scenario.

    This means: a scenario that clearly expresses 3 distinct needs will
    keep ~3 criteria (since they're all close to each other in relevance);
    a scenario that's really only about ONE thing will correctly collapse
    to 1 criterion (because everything else trails far behind the winner)
    - instead of always forcing exactly top_k criteria, or always
    collapsing to 1 regardless of how many things you actually asked for.

In [ ]:
def extract_requirements_semantic(domain: dict, user_scenario: str, top_k: int = 4,
                                   keyword_boost: float = 0.15, relative_floor: float = 0.55) -> dict:

    scenario_embedding = semantic_model.encode(user_scenario)
    similarities = {
        c: float(cosine_similarity([scenario_embedding], [emb])[0][0])
        for c, emb in domain["criterion_embeddings"].items()
    }

    text = user_scenario.lower()
    for criterion, keywords in CRITERION_BOOST_KEYWORDS.items():
        if criterion not in similarities:
            continue
        if any(kw in text for kw in keywords):
            similarities[criterion] += keyword_boost

    sorted_criteria = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
    candidates = sorted_criteria[:top_k]
    top_score = candidates[0][1]

    selected = {c: s for c, s in candidates if top_score > 0 and s >= relative_floor * top_score}
    if not selected:
        selected = {candidates[0][0]: candidates[0][1]}

    vals = list(selected.values())
    lo, hi = min(vals), max(vals)
    if hi > lo:
        selected = {c: round(0.3 + 0.7 * (v - lo) / (hi - lo), 3) for c, v in selected.items()}
    else:
        selected = {c: 0.8 for c in selected}

    return {c: selected.get(c, 0.0) for c in domain["criterion_descriptions"]}

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. EXPLANATION GENERATOR (markdown-formatted, friendly criterion names)
# ─────────────────────────────────────────────────────────────────────────────

FRIENDLY_NAMES = {
    "Reasoning": "Solving complex problems", "Coding": "Programming and debugging",
    "Mathematics": "Mathematical ability", "Summarization": "Understanding long documents",
    "Multilingual": "Working with multiple languages", "Long_Context": "Handling very large documents",
    "Creativity": "Creative writing and brainstorming", "Cost_Efficiency": "Keeping costs low",
    "Speed": "Fast responses", "Safety": "Reliable and trustworthy answers",
    "Prompt_Adherence": "Following your instructions accurately", "Photorealism": "Creating realistic images",
    "Artistic_Quality": "Producing artistic images", "Text_Rendering": "Generating readable text inside images",
    "Editing_Capability": "Editing existing images", "Commercial_Safety": "Safe for commercial use",
    "Motion_Quality": "Natural-looking motion", "Visual_Fidelity": "Overall video quality",
    "Consistency": "Keeping scenes and characters consistent", "Max_Duration": "Generating longer videos",
    "Naturalness": "Human-like voice", "Voice_Cloning": "Voice cloning capability",
    "Latency": "Quick voice response", "Emotional_Range": "Expressive speech",
}


def generate_explanation(domain, model, fuzzy_weights, ranking):
    dataset = domain["dataset"]
    active = sorted([(c, w) for c, w in fuzzy_weights.items() if w > 0], key=lambda x: x[1], reverse=True)

    lines = ["### Recommended Model\n", f"## {model}\n", "### Why was this model selected?\n",
              "Based on your request, these capabilities mattered most:\n"]
    for criterion, weight in active:
        lines.append(f"- **{FRIENDLY_NAMES.get(criterion, criterion)}** ({weight:.0%} importance)")
    lines.append("")
    lines.append(f"Among all available **{domain['name']}** models, **{model}** gives the strongest overall balance for these priorities.")

    strengths = [f"- Excellent at **{FRIENDLY_NAMES.get(c, c)}**." for c, w in active if dataset.loc[model, c] >= 0.80]
    if strengths:
        lines.append("\n### Main strengths\n")
        lines.extend(strengths)

    weak = [FRIENDLY_NAMES.get(c, c) for c, w in active if dataset.loc[model, c] < 0.50]
    if weak:
        lines.append("\n### Things to consider\n")
        lines.append("This model is the best overall choice, but is comparatively weaker in:")
        lines.extend(f"- {w}" for w in weak)

    if len(ranking) >= 2:
        runner = ranking.iloc[1]["Model"]
        lines.append(f"\n### Alternative\nIf your priorities change, **{runner}** would be the next best option.")

    return "\n".join(lines)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. MAIN DSS FUNCTION
# ─────────────────────────────────────────────────────────────────────────────

def intelligent_dss(user_scenario: str, top_k: int = 4, verbose: bool = True, force_domain: str = None):
    """Returns: (domain_name, ranking_df, fuzzy_weights, explanation_str, domain_similarities)"""
    routed_domain, domain_sims = route_domain(user_scenario)
    domain_name = force_domain if force_domain in DOMAINS else routed_domain
    domain = DOMAINS[domain_name]

    raw_scores = extract_requirements_semantic(domain, user_scenario, top_k=top_k)
    fuzzy_weights = fuzzify_weights(raw_scores)
    ranking = fuzzy_soft_decision(domain["dataset"], fuzzy_weights)
    best_model = ranking.iloc[0]["Model"]
    explanation = generate_explanation(domain, best_model, fuzzy_weights, ranking)

    if verbose:
        print("=" * 65)
        print("SCENARIO:", user_scenario.strip())
        print("\nDOMAIN ROUTING:")
        for d, s in sorted(domain_sims.items(), key=lambda x: x[1], reverse=True):
            marker = " <- selected" if d == domain_name else ""
            print(f"  {d:<24} {s:.4f}{marker}")
        print(f"\nUsing domain: {domain_name}")
        print("\nACTIVE CRITERIA -> FUZZY WEIGHTS:")
        for c, s in raw_scores.items():
            if s > 0:
                print(f"  {c:<20}  normalized_weight={fuzzy_weights[c]:.4f}")
        print(f"\nRECOMMENDED MODEL:  *  {best_model}  *\n")
        print(explanation)
        print("\nFULL RANKING:")
        print(ranking.to_string())
        print("=" * 65 + "\n")

    return domain_name, ranking, fuzzy_weights, explanation, domain_sims


if __name__ == "__main__":
    intelligent_dss(
        "I need an affordable AI model for software engineering, debugging complex "
        "bugs and solving logical programming problems quickly. Budget is tight."
    )
    intelligent_dss("I just want the absolute cheapest text AI available.")
    intelligent_dss(
        "I want to create a photorealistic product photo for an e-commerce listing, "
        "with accurate text on the packaging."
    )

SCENARIO: I need an affordable AI model for software engineering, debugging complex bugs and solving logical programming problems quickly. Budget is tight.

DOMAIN ROUTING:
  Text / Code LLMs         1.0000 <- selected
  Image Generation         0.0000
  Video Generation         0.0000
  Speech / Voice (TTS)     0.0000

Using domain: Text / Code LLMs

ACTIVE CRITERIA -> FUZZY WEIGHTS:
  Reasoning             normalized_weight=0.2621
  Coding                normalized_weight=0.6012
  Cost_Efficiency       normalized_weight=0.1367

RECOMMENDED MODEL:  *  GPT-5.5  *

### Recommended Model

## GPT-5.5

### Why was this model selected?

Based on your request, these capabilities mattered most:

- **Programming and debugging** (60% importance)
- **Solving complex problems** (26% importance)
- **Keeping costs low** (14% importance)

Among all available **Text / Code LLMs** models, **GPT-5.5** gives the strongest overall balance for these priorities.

### Main strengths

- Excellent at **Progra

In [ ]:
import io
import math
import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import gradio as gr
from PIL import Image
import markdown


# ----------------------------------------------------
# COLOR PALETTE
# ----------------------------------------------------

DOMAIN_COLORS = {
    "Text / Code LLMs": "#14B8A6",
    "Image Generation": "#8B5CF6",
    "Video Generation": "#F97316",
    "Speech / Voice (TTS)": "#3B82F6"
}

DOMAIN_ICONS = {
    "Text / Code LLMs":"💬",
    "Image Generation":"🎨",
    "Video Generation":"🎬",
    "Speech / Voice (TTS)":"🎤"
}

# ----------------------------------------------------
# CUSTOM CSS
# ----------------------------------------------------

css = """

body{

background:#F4F7FB;

font-family:Inter,sans-serif;

}


.gradio-container{

max-width:1450px !important;

margin:auto;

padding-top:20px;

background:#F4F7FB;

}


/* Remove footer */

footer{

display:none !important;

}


/* Main cards */

.gr-group,
.gr-box{

background:white !important;

border-radius:22px !important;

border:1px solid #E5E7EB !important;

box-shadow:
0px 10px 35px rgba(0,0,0,.06);

}


/* Hero */

.hero{

padding:35px;

border-radius:24px;

background:

linear-gradient(
135deg,
#0F766E,
#14B8A6
);

color:white;

margin-bottom:15px;

}


.hero h1{

font-size:42px;

margin-bottom:8px;

color:white;

}


.hero p{

font-size:18px;

opacity:.95;

}


/* metric cards */

.metric{

background:white;

border-radius:18px;

padding:18px;

text-align:center;

border:1px solid #E5E7EB;

box-shadow:0 5px 20px rgba(0,0,0,.05);

}


.metric h2{

margin:0;

font-size:34px;

}


.metric p{

margin:4px;

color:#64748B;

}


/* Recommendation card */

.recommend{

border-radius:22px;

padding:30px;

background:white;

border-left:8px solid #14B8A6;

box-shadow:0 10px 30px rgba(0,0,0,.08);

}

.recommend{
    color:#1F2937 !important;
}

.recommend h1{
    color:#111827 !important;
}

.recommend h2{
    color:#14B8A6 !important;
}

.recommend p{
    color:#374151 !important;
}

.recommend b{
    color:#111827 !important;
}


.recommend h1{

font-size:38px;

margin-bottom:10px;

}


.recommend h2{

color:#14B8A6;

margin-top:5px;

}


/* buttons */

button{

border:none !important;

background:

linear-gradient(
90deg,
#14B8A6,
#0F766E
) !important;

color:white !important;

font-weight:bold !important;

border-radius:14px !important;

transition:.3s;

}


button:hover{

transform:translateY(-2px);

box-shadow:0px 8px 20px rgba(20,184,166,.3);

}


/* textbox */

textarea{

border-radius:16px !important;

}


/* accordion */

details{

border-radius:15px;

}


/* table */

table{

border-collapse:collapse;

width:100%;

}

th{

background:#ECFDF5;

}

td,th{

padding:10px;

border:1px solid #E5E7EB;

}


/* scrollbar */

::-webkit-scrollbar{

width:8px;

}

::-webkit-scrollbar-thumb{

background:#14B8A6;

border-radius:20px;

}

"""

# ----------------------------------------------------
# HTML BUILDERS
# ----------------------------------------------------

def recommendation_card(domain,best_model,score):

    color=DOMAIN_COLORS[domain]
    icon=DOMAIN_ICONS[domain]

    return f"""

<div class="recommend">

<h2>{icon} Recommended Model</h2>

<h1>{best_model}</h1>

<p style="font-size:18px">

<b>Domain:</b>

<span style="color:{color}">
{domain}
</span>

</p>

<p style="font-size:18px">

<b>Overall Match Score:</b>

{score:.3f}

</p>

<p>

This model achieved the highest weighted fuzzy-soft-set
score after analysing your scenario, identifying the
most relevant criteria, and comparing all available
models using benchmark-informed scores.

</p>

</div>

"""

# ----------------------------------------------------

def metric_card(title,value):

    return f"""

<div class="metric">

<h2>{value}</h2>

<p>{title}</p>

</div>

"""

# ----------------------------------------------------

def weights_table(domain,fuzzy):

    benchmark=DOMAINS[domain]["benchmark_source"]

    rows=""

    ordered=sorted(
        fuzzy.items(),
        key=lambda x:x[1],
        reverse=True
    )

    for c,w in ordered:

        if w==0:
            continue

        rows+=f"""

<tr>

<td>{c.replace("_"," ")}</td>

<td>{w:.1%}</td>

<td>{benchmark[c]}</td>

</tr>

"""

    return f"""

<h3>Detected User Priorities</h3>

<table>

<tr>

<th>Criterion</th>

<th>Importance</th>

<th>Benchmark Source</th>

</tr>

{rows}

</table>

"""

# ----------------------------------------------------

def explanation_table(domain,best_model,fuzzy):

    dataset=DOMAINS[domain]["dataset"]

    benchmark=DOMAINS[domain]["benchmark_source"]

    rows=""

    ordered=sorted(
        fuzzy.items(),
        key=lambda x:x[1],
        reverse=True
    )

    for c,w in ordered:

        if w==0:
            continue

        score=dataset.loc[best_model,c]

        label=score_label(score)

        color="#10B981"

        if score<0.60:
            color="#EF4444"

        elif score<0.80:
            color="#F59E0B"

        rows+=f"""

<tr>

<td>{c.replace("_"," ")}</td>

<td>{w:.1%}</td>

<td>{score:.2f}</td>

<td>

<span style="color:{color};font-weight:bold">

{label}

</span>

</td>

<td>{benchmark[c]}</td>

</tr>

"""

    return f"""

<h3>Why this recommendation?</h3>

<table>

<tr>

<th>Criterion</th>

<th>Weight</th>

<th>Model Score</th>

<th>Rating</th>

<th>Evidence</th>

</tr>

{rows}

</table>

"""

# ----------------------------------------------------
# CHARTS
# ----------------------------------------------------

def ranking_chart(ranking,domain):

    color=DOMAIN_COLORS[domain]

    fig,ax=plt.subplots(figsize=(8,5))

    models=ranking["Model"][::-1]

    scores=ranking["Score"][::-1]

    bars=ax.barh(
        models,
        scores,
        color=color
    )

    ax.set_xlabel("Weighted Score")

    ax.set_title("Model Ranking")

    ax.grid(alpha=.2)

    for b,v in zip(bars,scores):

        ax.text(
            v+.01,
            b.get_y()+b.get_height()/2,
            f"{v:.2f}",
            va="center"
        )

    plt.tight_layout()

    buf=io.BytesIO()

    plt.savefig(
        buf,
        dpi=160,
        bbox_inches="tight"
    )

    plt.close()

    buf.seek(0)

    return Image.open(buf)

# ----------------------------------------------------

def radar_chart(domain,best_model):

    dataset=DOMAINS[domain]["dataset"]

    values=dataset.loc[best_model].values

    labels=list(dataset.columns)

    angles=np.linspace(
        0,
        2*np.pi,
        len(labels),
        endpoint=False
    )

    values=np.concatenate(
        (values,[values[0]])
    )

    angles=np.concatenate(
        (angles,[angles[0]])
    )

    fig=plt.figure(figsize=(6,6))

    ax=plt.subplot(
        111,
        polar=True
    )

    ax.plot(
        angles,
        values,
        linewidth=3
    )

    ax.fill(
        angles,
        values,
        alpha=.25
    )

    ax.set_ylim(0,1)

    ax.set_xticks(
        angles[:-1]
    )

    ax.set_xticklabels(
        [
            l.replace("_","\n")
            for l in labels
        ],
        fontsize=9
    )

    ax.set_title(
        best_model,
        pad=25,
        fontsize=15
    )

    plt.tight_layout()

    buf=io.BytesIO()

    plt.savefig(
        buf,
        dpi=160,
        bbox_inches="tight"
    )

    plt.close()

    buf.seek(0)

    return Image.open(buf)

# ----------------------------------------------------
# MAIN CALLBACK
# ----------------------------------------------------

def recommend(scenario,top_k):

    if scenario.strip()=="":
        return (
            "",
            "",
            "",
            "",
            None,
            None,
            ""
        )

    domain,ranking,fuzzy,explanation,domain_sim=intelligent_dss(
        scenario,
        top_k=int(top_k),
        verbose=False
    )

    best=ranking.iloc[0]["Model"]

    score=ranking.iloc[0]["Score"]

    card=recommendation_card(
        domain,
        best,
        score
    )

    metrics="""
<div style="display:grid;
grid-template-columns:repeat(4,1fr);
gap:15px;">
"""

    metrics+=metric_card(
        "Models Compared",
        len(ranking)
    )

    metrics+=metric_card(
        "Criteria Used",
        sum(
            w>0
            for w in fuzzy.values()
        )
    )

    metrics+=metric_card(
        "Winning Score",
        f"{score:.2f}"
    )

    metrics+=metric_card(
        "Detected Domain",
        DOMAIN_ICONS[domain]
    )

    metrics+="</div>"

    weight_html=weights_table(
        domain,
        fuzzy
    )

    explain_html=explanation_table(
        domain,
        best,
        fuzzy
    )

    rank_html=ranking.to_html(
        index=False,
        classes="table"
    )

    rank_chart=ranking_chart(
        ranking,
        domain
    )

    radar=radar_chart(
        domain,
        best
    )

    return(
        card,
        metrics,
        explain_html,
        weight_html,
        rank_chart,
        radar,
        rank_html
    )

# ----------------------------------------------------
# BUILD INTERFACE
# ----------------------------------------------------

theme=gr.themes.Base(
    primary_hue="emerald",
    secondary_hue="teal",
    neutral_hue="slate",
)

with gr.Blocks(
    theme=theme,
    css=css,
    title="AI Decision Support System"
) as demo:

    gr.HTML("""

<div class="hero">

<h1>🤖 Multi-Domain AI Decision Support System</h1>

<p>

Benchmark-informed recommendation engine powered by
Semantic Search + Fuzzy Soft Set Theory

</p>

</div>

""")

    with gr.Row():

        with gr.Column(scale=2):

            scenario=gr.Textbox(
                label="Describe your task",
                lines=8,
                placeholder="""Example:

• I need an affordable coding assistant

• Generate photorealistic product images

• Create cinematic AI videos

• Build a real-time voice assistant
"""
            )

            top_k=gr.Slider(
                2,
                6,
                value=4,
                step=1,
                label="Important Criteria"
            )

            submit=gr.Button(
                "🚀 Analyze Scenario",
                size="lg"
            )

        with gr.Column(scale=3):

            recommendation=gr.HTML()

            metrics=gr.HTML()
            recommendation = gr.HTML()

            metrics = gr.HTML()

    with gr.Accordion("📖 Why this recommendation?", open=True):

        explanation = gr.HTML()

    with gr.Accordion("🎯 Detected User Priorities", open=False):

        priorities = gr.HTML()

    with gr.Row():

        ranking_chart_output = gr.Image(
            label="Model Ranking",
            type="pil"
        )

        radar_chart_output = gr.Image(
            label="Capability Radar",
            type="pil"
        )

    with gr.Accordion("🏆 Leaderboard", open=False):

        leaderboard = gr.HTML()

    submit.click(

        fn=recommend,

        inputs=[

            scenario,

            top_k

        ],

        outputs=[

            recommendation,

            metrics,

            explanation,

            priorities,

            ranking_chart_output,

            radar_chart_output,

            leaderboard

        ]

    )

demo.launch()


/tmp/ipykernel_2258/1194430593.py:761: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://408b8dd49c68a01168.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
